In [ ]:
%cd ../..

In [ ]:
import awswrangler as wr
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter

In [ ]:
dr_pubs_grants = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/dr_pub_grant_links.xlsx')

In [ ]:
dr_schemes = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/award_list_legacy.xlsx')

In [ ]:
dr_schemes = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/DRscheme_mapping.xlsx',sheet_name='by_ref')

In [ ]:
lmics = pd.read_excel('notebooks/discovery_research/DR geographies_AAP list.xlsx', sheet_name='LMIC list')
lmics = lmics['LMIC_dimensions'].tolist()

In [ ]:
dr_pubs_grants = dr_pubs_grants.merge(dr_schemes[['Grant Reference','legacy','scheme']], how='left', left_on='Reference', right_on='Grant Reference')

In [ ]:
dr_pubs_grants.dropna(subset='legacy', inplace=True)

In [ ]:
subgroups = dr_pubs_grants.groupby('scheme')['id'].apply(lambda pub_ids: list(set(list(pub_ids)))).to_dict()

In [ ]:
pub_ids_l0 = list(set(dr_pubs_grants['id'].tolist()))

In [ ]:
subgroups.keys()

In [ ]:
subgroup = 'PhD'
pub_ids_l0 = subgroups[subgroup]

In [ ]:
len(pub_ids_l0)

In [ ]:
locations = Locations.from_publication_ids(pub_ids_l0, max_chunk_size=1000)

In [ ]:
locations_raw = locations.data

In [ ]:
locations.extract(location_level="institution")

In [ ]:
publication_level_data = locations.publication_data

In [ ]:
institution_lookup = locations.location_names

In [ ]:
locations.data = locations_raw
locations._clean_grid_ids()

In [ ]:
# How many authors with >1 grid id?
locations.data[locations.data['grid_id'].apply(lambda i: len(i))>1]

In [ ]:
locations.extract_locations()

In [ ]:
country_lookup = locations.location_names

In [ ]:
mapping = {}
for grid_id, country in country_lookup.items():
    if country=='United Kingdom':
        mapping[institution_lookup[grid_id]] = 'uk'
    else:
        if country in lmics:
            mapping[institution_lookup[grid_id]] = 'lmic'
        else:
            mapping[institution_lookup[grid_id]] = 'hic'

In [ ]:
mapping_reversed = {}
for grid_id, group in mapping.items():
    if mapping_reversed.get(group):
        mapping_reversed[group].append(grid_id)
    else:
        mapping_reversed[group] = [grid_id]

In [ ]:
def group_adjacency_matrices(adjacency_matrices):
    grouped_matrices = {}
    for year in adjacency_matrices.keys():
        adj_matrix = adjacency_matrices[year]['All']
        adj_matrix['total'] = adj_matrix.loc['total',:]
        adj_matrix['group'] = adj_matrix.index.map(mapping)
        adj_matrix.loc['All', 'group'] = 'All'
        adj_matrix.loc['total', 'group'] = 'total'
        gm = adj_matrix.groupby('group').sum()
        for k in mapping_reversed.keys():
            if k not in gm.index:
                gm.loc[k, :] = 0
        grouped_matrices[year] = gm
    return grouped_matrices

In [ ]:
grouped_adjacency_matrices = group_adjacency_matrices(locations.adjacency_matrices)

In [ ]:
uk_institutes = []
for year, am in grouped_adjacency_matrices.items():
    am = am[[c for c in am.columns if c in mapping_reversed['uk']]]
    if year>=2013:
        am = am.transpose().sort_values('lmic', ascending=False)
        am.reset_index(inplace=True)
        am.rename(columns={'l2':'institute'}, inplace=True)
        am['year'] = year
        for group in ['lmic','hic','uk']:
            am[f'{group}_pct'] = am[group] / am['All'] *100
        uk_institutes.append(am)
uk_institutes = pd.concat(uk_institutes)

In [ ]:
lmic_institutes = []
for year, am in grouped_adjacency_matrices.items():
    am = am[[c for c in am.columns if c in mapping_reversed['lmic']]]
    if year>=2013:
        am = am.transpose().sort_values('uk', ascending=False)
        am['year'] = year
        am.reset_index(inplace=True)
        am.rename(columns={'l2':'institute'}, inplace=True)
        for group in ['lmic','hic','uk']:
            am[f'{group}_pct'] = am[group] / am['All'] *100
        lmic_institutes.append(am)
lmic_institutes = pd.concat(lmic_institutes)

In [ ]:
uk_all = uk_institutes.groupby('institute').sum().drop('year', axis=1).reset_index().sort_values('lmic', ascending=False)
lmic_all = lmic_institutes.groupby('institute').sum().drop('year', axis=1).reset_index().sort_values('uk', ascending=False)

In [ ]:
top30_uk = list(reversed(uk_all.head(30)['institute'].tolist()))

In [ ]:
top30_lmic = list(reversed(lmic_all.head(30)['institute'].tolist()))

In [ ]:
#uk_institutes.to_excel('notebooks/discovery_research/uk_institutions.xlsx', sheet_name='uk_institutes', index=False)

In [ ]:
#lmic_institutes.to_excel('notebooks/discovery_research/lmic_institutions.xlsx', sheet_name='lmic_institutes', index=False)

In [ ]:
uk_data = {}
uk_data_pct = {}
for year in uk_institutes['year'].unique():
    df = uk_institutes[(uk_institutes['year']==year) & (uk_institutes['institute'].isin(top30_uk))]
    for i in set(top30_uk).difference(set(df['institute'].tolist())):
        df = pd.concat([df, pd.DataFrame({'institute':[i],
                                     'All':[0],
                                     'lmic':[0],
                                     'hic':[0],
                                     'total':[0],
                                     'uk':[0],
                                     'year':[year],
                                     'lmic_pct':[0],
                                     'hic_pct':[0],
                                     'uk_pct':[0]})])
    sort_order = {inst: order for order, inst in enumerate(top30_uk)}
    df['sort_order'] = df['institute'].map(sort_order)
    df = df.sort_values(by='sort_order')
    uk_data[str(year)] = [
        {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]
    uk_data_pct[str(year)] = [
        {
            'x': df['lmic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['uk_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/uk_institutions{"_"+subgroup}.json','w')) as f:
    json.dump(uk_data, f)

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/uk_institutions_pct{"_"+subgroup}.json','w')) as f:
    json.dump(uk_data_pct, f)

In [ ]:
lmic_institutes.head()

In [ ]:
lmic_data = {}
lmic_data_pct = {}
for year in lmic_institutes['year'].unique():
    df = lmic_institutes[(lmic_institutes['year']==year) & (lmic_institutes['institute'].isin(top30_lmic))]
    for i in set(top30_lmic).difference(set(df['institute'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'institute':[i],
                                     'All':[0],
                                     'lmic':[0],
                                     'hic':[0],
                                     'total':[0],
                                     'uk':[0],
                                     'year':[year],
                                     'lmic_pct':[0],
                                     'hic_pct':[0],
                                     'uk_pct':[0]})])
    sort_order = {inst: order for order, inst in enumerate(top30_lmic)}
    df['sort_order'] = df['institute'].map(sort_order)
    df = df.sort_values(by='sort_order')
    lmic_data[str(year)] = [
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]
    lmic_data_pct[str(year)] = [
        {
            'x': df['uk_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['lmic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/lmic_institutions{"_"+subgroup}.json','w')) as f:
    json.dump(lmic_data, f)

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/lmic_institutions_pct{"_"+subgroup}.json','w')) as f:
    json.dump(lmic_data_pct, f)

In [ ]:
if subgroup != 'All':
    grants_info = dr_pubs_grants[dr_pubs_grants['scheme']==subgroup]
else:
    grants_info = dr_pubs_grants

In [ ]:
grants_df = locations.publication_data.merge(grants_info[['id','Reference']], left_on='dimensions_publication_id', right_on='id')

In [ ]:
mapping = {}
for country in country_lookup.values():
    if country == 'United Kingdom':
        mapping[country] = 'uk'
    elif country in lmics:
        mapping[country] = 'lmic'
    else:
        mapping[country] = 'hic'

In [ ]:
grants_df['group'] = grants_df['location'].map(mapping)

In [ ]:
lmic_wellcome_authorships = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        lmic_wellcome_authorships[year]=df[df['group']=='lmic'].groupby('location')['Reference'].count().reset_index()

In [ ]:
lmic_wellcome_all = grants_df[grants_df['group']=='lmic'].groupby('location')['Reference'].count().sort_values(ascending=False)

In [ ]:
lmic_wellcome_top30 = list(reversed(lmic_wellcome_all.reset_index().head(30)['location'].tolist()))

In [ ]:
wellcome_authorships_data = {}
for year, counts in lmic_wellcome_authorships.items():
    df = counts[counts['location'].isin(lmic_wellcome_top30)]
    for i in set(lmic_wellcome_top30).difference(set(df['location'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'location':[i],
                                     'Reference':[0]})])
    sort_order = {inst: order for order, inst in enumerate(lmic_wellcome_top30)}
    df['sort_order'] = df['location'].map(sort_order)
    df = df.sort_values(by='sort_order')
    wellcome_authorships_data[str(year)] = [
            {
                'x': df['Reference'].astype(int).tolist(),
                'y': df['location'].tolist(),
                'name': 'Number of authorships',
                'orientation': 'h',
                'marker': {
                    'color': 'rgba(21,113,242,0.7)',
                    'width': 1
                },
                'type': 'bar'
            }
        ]

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/lmic_wellcome_authorships{"_"+subgroup}.json','w')) as f:
    json.dump(wellcome_authorships_data, f)

In [ ]:
lmic_unique_grant_authorships = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        lmic_unique_grant_authorships[year] = df[df['group']=='lmic'].groupby('location')['Reference'].nunique().reset_index()

In [ ]:
lmic_grant_all = grants_df[grants_df['group']=='lmic'].groupby('location')['Reference'].nunique().sort_values(ascending=False)

In [ ]:
lmic_grant_top30 = list(reversed(lmic_grant_all.reset_index().head(30)['location'].tolist()))

In [ ]:
grant_authorships_data = {}
for year, counts in lmic_unique_grant_authorships.items():
    df = counts[counts['location'].isin(lmic_grant_top30)]
    for i in set(lmic_grant_top30).difference(set(df['location'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'location':[i],
                                     'Reference':[0]})])
    sort_order = {inst: order for order, inst in enumerate(lmic_grant_top30)}
    df['sort_order'] = df['location'].map(sort_order)
    df = df.sort_values(by='sort_order')
    grant_authorships_data[str(year)] = [
            {
                'x': df['Reference'].astype(int).tolist(),
                'y': df['location'].tolist(),
                'name': 'Number of unique grants',
                'orientation': 'h',
                'marker': {
                    'color': 'rgba(21,113,242,0.7)',
                    'width': 1
                },
                'type': 'bar'
            }
        ]

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/lmic_grant_authorships{"_"+subgroup}.json','w')) as f:
    json.dump(grant_authorships_data, f)

In [ ]:
grant_titles = pd.read_excel('notebooks/discovery_research/DR_overview_dash_export.xlsx', sheet_name='Overview')

In [ ]:
grant_titles = grant_titles.set_index('Reference')['Title'].to_dict()

In [ ]:
grants_df['grant_title'] = grants_df['Reference'].map(grant_titles)
#grants_df.to_excel('notebooks/discovery_research/grants.xlsx', sheet_name='grants', index=False)

In [ ]:
#grants_df[grants_df['group']=='china']['grant_title'].value_counts().to_csv('notebooks/discovery_research/china_grants.csv')

In [ ]:
#grants_df[grants_df['group']=='china']

In [ ]:
grant_matrices = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        am = pd.crosstab(df['Reference'], df['group']).reset_index()
        am['title'] = am['Reference'].map(grant_titles)
        grant_matrices[year] = am

In [ ]:
grant_lmic_data = {}
for year, df in grant_matrices.items():
    df = df.sort_values(by='lmic', ascending=False).head(30)
    df = df.reset_index()[::-1]
    grant_lmic_data[str(year)] = [
            {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        },
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        }
        ]

In [ ]:
with(open(f'notebooks/discovery_research/barcharts/grant_lmic_data{"_"+subgroup}.json','w')) as f:
    json.dump(grant_lmic_data, f)

In [ ]:
locations.data = locations_raw
locations.extract_locations(location_level='institution')
institution_lookup = locations.location_names

In [ ]:
locations.data = locations_raw
locations.extract_edges()

In [ ]:
def convert_edges(coauthorship_edges, country_names, institution_names, lmics):
    converted_edges = {}
    for year, coauthorship_edges in coauthorship_edges.items():
        converted_edges[year] = Counter()
        for edge, weight in coauthorship_edges.items():
            grid_id_0, grid_id_1 = edge
            country_0 = country_names.get(grid_id_0)
            country_1 = country_names.get(grid_id_1)
            if (country_0=='United Kingdom' and (country_1 in lmics)):
                converted_edges[year].update({(institution_names.get(grid_id_0), country_1): weight})
            elif (country_1=='United Kingdom' and (country_0 in lmics)):
                converted_edges[year].update({(institution_names.get(grid_id_1), country_0): weight})
    return converted_edges

In [ ]:
uk_lmic_edges = convert_edges(locations.coauthorship_edges, country_lookup, institution_lookup, lmics)

In [ ]:
locations.publication_data

In [ ]:
pub_ids_l0 = list(set(dr_pubs_grants['id'].tolist()))

In [ ]:
locations = Locations.from_publication_ids(pub_ids_l0, max_chunk_size=1000)

In [ ]:
locations.extract()

In [ ]:
locations.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.001, directed=False, threshold=0, node_count="total", font = {"size": 20, "face": "Helvetica Neue"})

In [ ]:
locations.to_visjs(vis_name="locations", directed=False)

In [ ]:
locations._to_json("nodes_all.json", locations.vis_nodes)
locations._to_json("edges_all.json", locations.vis_edges)

Add subgroups

In [ ]:
for subgroup, pub_ids in subgroups.items():
    print(subgroup)
    locations = Locations.from_publication_ids(pub_ids, max_chunk_size=1000)
    locations.extract()
    locations.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.001, directed=False, threshold=0, node_count="total", font = {"size": 20, "face": "Helvetica Neue"})
    locations._to_json(f"nodes_{subgroup}.json", locations.vis_nodes)
    locations._to_json(f"edges_{subgroup}.json", locations.vis_edges)

Add positions based on latitude and longitude

In [ ]:
lat_lon = pd.read_csv('notebooks/discovery_research/countries.csv')

In [ ]:
lat_lon.rename(columns={'Latitude (average)':'y', 'Longitude (average)':'x'}, inplace=True)

In [ ]:
lat_lon.dropna(subset=['x','y'], inplace=True)

In [ ]:
lat_lon['y'] = lat_lon['y']*-1

In [ ]:
lat_lon['x'] = lat_lon['x']*25
lat_lon['y'] = lat_lon['y']*25

In [ ]:
positions = lat_lon.set_index('Country')[['x','y']].to_dict('index')

In [ ]:
with(open('notebooks/discovery_research/positions.json','w')) as f:
    json.dump(positions, f)

In [ ]:
for c in set(locations.location_names.values()):
    if c not in positions.keys():
        print(c)

In [ ]:
#wr.s3.to_parquet(locations.publication_data, 's3://datalabs-data/funding_impact_measures/dr_publications/dr_pubs_countries.parquet')

In [ ]:
with(open('notebooks/location_vis/edges.json','r')) as f:
    edges = json.load(f)

In [ ]:
def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict

In [ ]:
edge_fnames = ['edges_all.json', 'edges_cd.json', 'edges_da.json', 'edges_dir.json', 'edges_ec.json', 'edges_other.json']

In [ ]:
for sg in subgroups.keys():
    with(open(f'notebooks/location_vis//new/edges_{sg}.json','r')) as f:
        edges = json.load(f)
        for e in edges:
            weight = e['title'].split(': ')[-1].replace(',','')
            e['weight'] = int(weight)
    edge_dict = edges_to_dict(edges)
    with(open(f'notebooks/location_vis/new/dict_edges_{sg}.json','w')) as f:
        json.dump(edge_dict, f)

In [ ]:
node_fnames = ['nodes_all.json', 'nodes_cd.json', 'nodes_da.json', 'nodes_dir.json', 'nodes_ec.json', 'nodes_other.json']

In [ ]:
with(open(f'notebooks/location_vis/{node_fnames[0]}','r')) as f:
    nodes = json.load(f)

In [ ]:
min_size = 5

In [ ]:
for data in nodes.values():
    for d in data:
        d['size'] += (min_size-1)

In [ ]:
# Adjust node size
for sg in subgroups.keys():
    with(open(f'notebooks/location_vis/new/nodes_{sg}.json','r')) as f:
        nodes = json.load(f)
    for data in nodes.values():
        for d in data:
            d['size'] += (min_size-1)
    with(open(f'notebooks/location_vis/new/nodes_{sg}.json','w')) as f:
        json.dump(nodes, f)

In [ ]:
with(open('notebooks/discovery_research/edges.json','w')) as f:
    json.dump(edges, f)